In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import time

# ==============================================================================
# 1. ENVIRONMENT CONFIGURATION
# ==============================================================================

# Force reloading of environment variables to ensure local runtime context shifts
# or secrets rotation are immediately captured without restarting the process.
load_dotenv(override=True)

# ==============================================================================
# 2. DATABASE CONNECTION SETUP
# ==============================================================================

# Construct standard connection URI string for PostgreSQL dialect.
# SQLAlchemy engine decouples connection management and provisions a thread-safe connection pool.
DATABASE_URL = f"postgresql://{os.getenv('user')}:{os.getenv('password')}@{os.getenv('host')}:{os.getenv('port')}/{os.getenv('dbname')}"
engine = create_engine(DATABASE_URL)

# ==============================================================================
# 3. DATA INGESTION, TRANSFORMATION, AND LOADING
# ==============================================================================
try:
    print("Loading CSV into memory...")
    
    # Ingest historical wide/long metrics payload from the local extraction stage
    df = pd.read_csv('../1_data_extraction/data/historical_assets.csv')

    # Normalize column headers to lowercase to prevent string-matching friction 
    # against lowercase-native PostgreSQL identifiers
    df.columns = df.columns.str.lower()

    # Map Pandas DataFrame column labels to match destination relational database column definitions
    df = df.rename(columns={
        'ticker': 'ticker',
        'date': 'date',
        'open': 'open_price',
        'high': 'high_price',
        'low': 'low_price',
        'close': 'close_price',
        'volume': 'volume',
        'return_3m': 'return_3m',
        'return_6m': 'return_6m',
        'momentum_score': 'momentum_score',
        'returns': 'returns',
        'volatility': 'volatility',
        'low_vol_score': 'low_vol_score'
    })
    
    # Convert dates to datetime.date format to ensure compatibility with SQL DATE types 
    # and prevent timestamp/timezone serialization issues during serialization
    df['date'] = pd.to_datetime(df['date']).dt.date
    
    # Project and slice only the columns defined in the target database schema,
    # filtering out any temporary runtime metrics or unindexed source metadata fields
    valid_columns = [
        'ticker', 'date', 'open_price', 'high_price', 'low_price', 'close_price', 
        'volume', 'return_3m', 'return_6m', 'momentum_score', 'returns', 'volatility', 'low_vol_score'
    ]
    df_final = df[valid_columns]

    # Batch Processing Configuration
    # A chunk size of 500 balances database memory overhead and network roundtrips,
    # preventing internal connection pooler saturation (e.g., PgBouncer transaction limits).
    total_rows = len(df_final)
    chunk_size = 500 

    print(f"Starting upload of {total_rows} rows in chunks of {chunk_size}...")
    start_time = time.time()

    # Bulk load finalized structured elements into the relational data store.
    # 'method=multi' expands inserts into multiple-row parameter clauses for major speed gains.
    df_final.to_sql(
        'price_history', 
        engine, 
        if_exists='append', 
        index=False, 
        method='multi', 
        chunksize=chunk_size
    )

    end_time = time.time()
    duration = (end_time - start_time) / 60
    print(f"\nUpload completed successfully in {duration:.2f} minutes!")

except Exception as e:
    # Catch-all block to isolate network drops, memory limits, or constraint violations,
    # preserving data pipeline failure context without masking the source trace.
    print(f"\nCritical Error: {e}")

Loading CSV into memory...
Starting upload of 343590 rows in chunks of 500...

Upload completed successfully in 2.76 minutes!
